# Аугментация данных

In [11]:
import cv2
import numpy as np
import random

# Настройки путей и объёма данных
IMAGE_PATH = "input.jpeg"
MASK_PATH = "base_mask.png"
OUTPUT_DIR = "dataset"

img = cv2.imread(IMAGE_PATH)
mask = cv2.imread(MASK_PATH, cv2.IMREAD_GRAYSCALE)

# Применения различных искажений к фото

In [13]:
for i in range(45):
    aug_img = img.copy()
    aug_mask = mask.copy()

    # 1. Отражение
    if random.random() > 0.5:
        flip_code = random.choice([1, 0, -1])
        aug_img = cv2.flip(aug_img, flip_code)
        aug_mask = cv2.flip(aug_mask, flip_code)

    # 2. Поворот без потери границ
    if random.random() > 0.3:
        angle = random.randint(-45, 45)

        h, w = aug_img.shape[:2]
        cX, cY = w // 2, h // 2

        M = cv2.getRotationMatrix2D((cX, cY), angle, 1.0)
        cos = np.abs(M[0, 0])
        sin = np.abs(M[0, 1])

        nW = int((h * sin) + (w * cos))
        nH = int((h * cos) + (w * sin))

        M[0, 2] += (nW / 2) - cX
        M[1, 2] += (nH / 2) - cY

        aug_img = cv2.warpAffine(
            aug_img, M, (nW, nH),
            flags=cv2.INTER_LINEAR,
            borderMode=cv2.BORDER_CONSTANT,
            borderValue=(128, 128, 128)
        )

        aug_mask = cv2.warpAffine(
            aug_mask, M, (nW, nH),
            flags=cv2.INTER_NEAREST,
            borderMode=cv2.BORDER_CONSTANT,
            borderValue=0
        )

    # 3. Изменение цвета и яркости
    if random.random() > 0.4:
        hsv = cv2.cvtColor(aug_img, cv2.COLOR_BGR2HSV).astype(np.float64)
        hsv[:, :, 1] *= random.uniform(0.8, 1.2)
        hsv[:, :, 2] += random.randint(-25, 25)
        hsv = np.clip(hsv, 0, 255).astype(np.uint8)
        aug_img = cv2.cvtColor(hsv, cv2.COLOR_HSV2BGR)

    # 4. Размытие
    if random.random() > 0.6:
        kernel_size = random.choice([3, 5, 7])
        aug_img = cv2.GaussianBlur(aug_img, (kernel_size, kernel_size), 0)

    # 5. Эффект
    if random.random() > 0.7:
        prob = 0.02
        thres = 1 - prob
        for row in range(aug_img.shape[0]):
            for col in range(aug_img.shape[1]):
                rdn = random.random()
                if rdn < prob:
                    aug_img[row, col] = [0, 0, 0]
                elif rdn > thres:
                    aug_img[row, col] = [255, 255, 255]

    # 6. Гауссовский цифровой шум
    if random.random() > 0.7:
        mean = 0
        sigma = random.randint(5, 15)
        gauss_noise = np.random.normal(mean, sigma, aug_img.shape).astype("int16")
        aug_img = aug_img.astype("int16") + gauss_noise
        aug_img = np.clip(aug_img, 0, 255).astype("uint8")

    # 7. Эффект  засветки
    if random.random() > 0.7:
        exposure_factor = random.uniform(1.2, 1.4)
        aug_img = np.clip(aug_img.astype(np.float64) * exposure_factor, 0, 255).astype(np.uint8)

    # 8. Эффект теплого/холодного света
    if random.random() > 0.6:
        is_warm = random.choice([True, False])

        b, g, r = cv2.split(aug_img)

        if is_warm:
            r = np.clip(r.astype(np.int16) + 20, 0, 255).astype(np.uint8)
            g = np.clip(g.astype(np.int16) + 10, 0, 255).astype(np.uint8)
            b = np.clip(b.astype(np.int16) - 10, 0, 255).astype(np.uint8)
        else:
            b = np.clip(b.astype(np.int16) + 25, 0, 255).astype(np.uint8)
            r = np.clip(r.astype(np.int16) - 15, 0, 255).astype(np.uint8)
            g = np.clip(g.astype(np.int16) - 5, 0, 255).astype(np.uint8)

        # Собираем каналы обратно в цветное изображение
        aug_img = cv2.merge([b, g, r])

    img_filename = f"{OUTPUT_DIR}/{i:03d}_image.jpg"
    mask_filename = f"{OUTPUT_DIR}/{i:03d}_mask.png"

    cv2.imwrite(img_filename, aug_img)
    cv2.imwrite(mask_filename, aug_mask)